In [ ]:
import pandas as pd
import numpy as np
import time
import psutil
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_PATH = "/content/drive/My Drive/Datasets/Customer_Seg/"

In [ ]:
customers = pd.read_csv(f"{BASE_PATH}/olist_customers_dataset.csv")
orders = pd.read_csv(f"{BASE_PATH}/olist_orders_dataset.csv")
items = pd.read_csv(f"{BASE_PATH}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{BASE_PATH}/olist_order_payments_dataset.csv")
Reviews = pd.read_csv(f"{BASE_PATH}/olist_order_reviews_dataset.csv")

In [ ]:
# Convert date columns
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [ ]:
# --- 2. Link Orders to Unique IDs immediately ---
# This ensures every order has the permanent 'customer_unique_id'
orders = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')

In [ ]:
# We use the last 90 days as our "test" period to see who churns
last_date = orders['order_purchase_timestamp'].max()
snapshot_date = last_date - pd.Timedelta(days=90)

# Historical data (The "Past" used for features)
hist_orders = orders[orders['order_purchase_timestamp'] < snapshot_date].copy()
# Future data (The "Future" used for the churn label)
future_orders = orders[orders['order_purchase_timestamp'] >= snapshot_date].copy()

In [ ]:
# Step3 -Feature Engineering (The "Signals") ---
print("Engineering features...")

# Feature A: Average Review Score per Customer

cust_reviews = Reviews.merge(orders[['order_id', 'customer_unique_id']], on='order_id')
hist_reviews = cust_reviews[cust_reviews['order_id'].isin(hist_orders['order_id'])]
avg_reviews = hist_reviews.groupby('customer_unique_id')['review_score'].mean().reset_index()

# Feature B: Delivery Performance (Late Deliveries)
hist_orders['is_late'] = (hist_orders['order_delivered_customer_date'] >
                          hist_orders['order_estimated_delivery_date']).astype(int)
delivery_stats = hist_orders.groupby('customer_unique_id')['is_late'].mean().reset_index()

# Feature C: RFM (Using the Snapshot Date as 'Today')
rfm = hist_orders.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days, # Recency
    'order_id': 'count' # Frequency
}).reset_index()
rfm.columns = ['customer_unique_id', 'recency', 'frequency']

Engineering features...


In [ ]:
# --- 4. Define the Target (Actual Churn) ---
# A customer churned if they appear in 'hist_orders' but NOT in 'future_orders'
active_in_future = future_orders['customer_unique_id'].unique()
rfm['is_churned'] = rfm['customer_unique_id'].apply(lambda x: 1 if x not in active_in_future else 0)

In [ ]:
# --- 5. Merge Features ---
final_df = rfm.merge(avg_reviews, on='customer_unique_id', how='left')
final_df = final_df.merge(delivery_stats, on='customer_unique_id', how='left')
final_df = final_df.fillna({'review_score': 3, 'is_late': 0}) # Fill missing values

# Prepare X and y
X = final_df[['recency', 'frequency', 'review_score', 'is_late']]
y = final_df['is_churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
OUTPUT_PATH = f"{BASE_PATH}"

import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
# Combine features + target for saving
train_df = pd.concat([
    pd.DataFrame(X_train, columns=X.columns),
    pd.Series(y_train.values, name="Churn")
], axis=1)

test_df = pd.concat([
    pd.DataFrame(X_test, columns=X.columns),
    pd.Series(y_test.values, name="Churn")
], axis=1)

# Save to CSV
train_df.to_csv(f"{OUTPUT_PATH}/train_dataset.csv", index=False)
test_df.to_csv(f"{OUTPUT_PATH}/test_dataset.csv", index=False)



In [ ]:
def save_predictions(model, X_te, y_te, model_name):
    preds = model.predict(X_te)

    output_df = pd.DataFrame({
        "Actual": y_te,
        "Predicted": preds
    })

    file_path = f"{OUTPUT_PATH}/{model_name}_churn_predictions.csv"
    output_df.to_csv(file_path, index=False)

    print(f"{model_name} predictions saved to {file_path}")


In [ ]:
# --- 6. Train Models with Resource Monitoring ---
def monitor_and_train(model, name, X_tr, y_tr):
    start_time = time.time()
    start_cpu = psutil.cpu_percent(interval=None)

    model.fit(X_tr, y_tr)

    end_time = time.time()
    end_cpu = psutil.cpu_percent(interval=None)

    print(f"\n--- {name} Training Finished ---")
    print(f"Time: {end_time - start_time:.4f}s")
    print(f"CPU Usage: {end_cpu}%")
    return model

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
monitor_and_train(rf, "Random Forest", X_train_scaled, y_train)

# Train XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
monitor_and_train(xgb, "XGBoost", X_train_scaled, y_train)


--- Random Forest Training Finished ---
Time: 4.1807s
CPU Usage: 72.1%


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [01:13:51] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- XGBoost Training Finished ---
Time: 0.2263s
CPU Usage: 100.0%


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
# --- 7. Evaluation ---
def evaluate(model, X_te, y_te, name):
    preds = model.predict(X_te)
    print(f"\nResults for {name}:")
    print(f"Accuracy: {accuracy_score(y_te, preds):.4f}")
    print(classification_report(y_te, preds))

evaluate(rf, X_test_scaled, y_test, "Random Forest")
evaluate(xgb, X_test_scaled, y_test, "XGBoost")


Results for Random Forest:
Accuracy: 0.9970
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        50
           1       1.00      1.00      1.00     17335

    accuracy                           1.00     17385
   macro avg       0.50      0.50      0.50     17385
weighted avg       0.99      1.00      1.00     17385


Results for XGBoost:
Accuracy: 0.9971
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        50
           1       1.00      1.00      1.00     17335

    accuracy                           1.00     17385
   macro avg       0.50      0.50      0.50     17385
weighted avg       0.99      1.00      1.00     17385



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
save_predictions(rf, X_test_scaled, y_test, "Random_Forest")

Random_Forest predictions saved to /content/drive/My Drive/Datasets/Customer_Seg//Random_Forest_churn_predictions.csv


In [ ]:
save_predictions(xgb, X_test_scaled, y_test, "XGBoost")


XGBoost predictions saved to /content/drive/My Drive/Datasets/Customer_Seg//XGBoost_churn_predictions.csv


In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)


LogisticRegression(max_iter=1000)

In [ ]:
evaluate(log_reg, X_test_scaled, y_test, "Logistic Regression")



Results for Logistic Regression:
Accuracy: 0.9971
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        50
           1       1.00      1.00      1.00     17335

    accuracy                           1.00     17385
   macro avg       0.50      0.50      0.50     17385
weighted avg       0.99      1.00      1.00     17385



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
save_predictions(log_reg, X_test_scaled, y_test, "Logistic_Regression")


Logistic_Regression predictions saved to /content/drive/My Drive/Datasets/Customer_Seg//Logistic_Regression_churn_predictions.csv


In [ ]:
import tensorflow

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

nn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
nn_model.fit(
    X_train_scaled, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9921 - loss: 0.0788 - val_accuracy: 0.9976 - val_loss: 0.0222
Epoch 2/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9973 - loss: 0.0203 - val_accuracy: 0.9976 - val_loss: 0.0193
Epoch 3/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9974 - loss: 0.0200 - val_accuracy: 0.9976 - val_loss: 0.0184
Epoch 4/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9970 - loss: 0.0219 - val_accuracy: 0.9976 - val_loss: 0.0183
Epoch 5/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9974 - loss: 0.0195 - val_accuracy: 0.9976 - val_loss: 0.0181
Epoch 6/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9976 - loss: 0.0181 - val_accuracy: 0.9976 - val_loss: 0.0183
Epoch 7/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9971 - loss: 0.0212 - val_accuracy: 0.9976 - val_loss: 0.0188
Epoch 8/20
1739/1739 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9973 - loss: 0.0192 - 

In [ ]:
nn_preds = (nn_model.predict(X_test_scaled) > 0.5).astype(int).ravel()

nn_output = pd.DataFrame({
    "Actual": y_test,
    "Predicted": nn_preds
})

nn_output.to_csv(
    f"{OUTPUT_PATH}/Neural_Network_churn_predictions.csv",
    index=False
)

print("Neural Network predictions saved")


544/544 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step
Neural Network predictions saved
